#Ingest drivers.json file
1. Read the file using Spark DataFrame reader API
2. Define and enforce Schema(preserve the nested structure)
3. Add Metadata Columns
    - Source File
    - Ingestion TimeStamp
4. Write to bronze delta table

In [0]:
%run ../00-common/01.environment_config

In [0]:
%run ../00-common/02.bronze-helpers

In [0]:
source_path = f"{landing_folder_path}/drivers.json"
table_name = f"{catalog_name}.{bronze_schema}.drivers"

### Step 1& 2- Read the json file using the DataFrame Reader API and enforce the schema

In [0]:
from pyspark.sql.types import StructType,StructField,StringType,DoubleType,DateType

name_schema = StructType([
                    StructField("givenName",StringType()),
                    StructField("familyName",StringType())                   
                                            ])
drivers_schema = StructType([
            StructField('driverId',StringType()),
            StructField('name',name_schema),
            StructField('dateOfBirth',DateType()),
            StructField('nationality',StringType()),
            StructField('url',StringType())
                                                ])



In [0]:
drivers_df = (
    spark.read
         .format("json")
         .option("header","true")
#        .option("inferSchema","true")
         .schema(drivers_schema)
         .option("mode","FAILFAST")
         .load(source_path)
        )


In [0]:
display(drivers_df)

###Step 3- Add Metadata Columns
- Source File
- Ingestion TimeStamp



In [0]:

drivers_final_df = add_ingestion_metadata(drivers_df)

In [0]:
display(drivers_final_df)

### Step 3- Write to bronze delta table

In [0]:
(
    drivers_final_df
        .write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(table_name)
)

In [0]:
display(spark.table(table_name))